#  Connexion à PostgreSQL

In [ ]:
from sqlalchemy import create_engine
import pandas as pd
from dotenv import load_dotenv
import os

# Charger les variables depuis le fichier .env
load_dotenv()

# Lire les variables d'environnement
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# Créer la connexion à la base PostgreSQL
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Préparation des données pour visualisation

Lire les tables nécessaires pour le dashboard

In [ ]:

products_df= pd.read_sql("SELECT * FROM products", engine)
df_customers = pd.read_sql("SELECT * FROM customers", engine)
df_regions = pd.read_sql("SELECT * FROM regions", engine)
df_orders = pd.read_sql("SELECT * FROM orders", engine)

Vérifier la cohérence des données
Types de données corrects

In [ ]:
print(df_orders.dtypes)
print(products_df.dtypes)
print(df_customers.dtypes)
print(df_regions.dtypes)

In [ ]:
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'], errors='coerce')
df_orders['ship_date'] = pd.to_datetime(df_orders['ship_date'], errors='coerce')


In [ ]:
print(df_regions.dtypes)

 Vérifier la complétude (valeurs manquantes)

In [ ]:
print("Valeurs nulles dans orders_df :\n", df_orders.isnull().sum())
print("Valeurs nulles dans products_df :\n", products_df.isnull().sum())
print("Valeurs nulles dans customers_df :\n", df_customers.isnull().sum())
print("Valeurs nulles dans regions_df :\n", df_regions.isnull().sum())

## Calculer des métriques clés

Total Sales per product

In [ ]:
sales_per_product = products_df.groupby('product_name')['sales'].sum().reset_index()
sales_per_product

Total Sales per category

In [ ]:
sales_per_category = products_df.groupby('category')['sales'].sum().reset_index()
sales_per_category

Total Sales per region

In [ ]:
orders_with_sales = df_orders.merge(
    products_df[['product_name','sales']], 
    on='product_name', 
    how='left'
)

sales_per_region = orders_with_sales.merge(
    df_regions[['customer_id','region_name']], 
    on='customer_id', 
    how='left'
)

sales_per_region = sales_per_region.groupby('region_name')['sales'].sum().reset_index()
sales_per_region

Total Profit

In [ ]:
total_profit = products_df['marge'].sum()
print(total_profit)

Profit Margin

In [ ]:
profit_margin= (products_df['marge'].sum() / products_df['sales'].sum()) * 100
profit_margin

In [ ]:
avg_profit_margin = profit_margin.mean()
print(f'{avg_profit_margin:.0f}')


Quantité vendue par catégorie

In [ ]:
orders_products = df_orders.merge(
    products_df[['product_name','category']],
    on='product_name',
    how='left'
)

quantity_per_category = orders_products.groupby('category')['order_id'].count().reset_index()
quantity_per_category

Calculer statistiques de base 

In [ ]:
stats_sales = products_df[['sales', 'marge']].agg(['mean', 'median', 'min', 'max', 'std'])
stats_sales

##  Feature engineering pour visualisation

colonnes dérivées mois-année

In [ ]:

df_orders['month_year'] = df_orders['order_date'].dt.to_period('M')
df_orders['month_year'] 

Profit Ratio

In [ ]:
products_df['profit_ratio'] = products_df['marge'] / products_df['sales']
products_df['profit_ratio'] 

# Agrégation des données 

Sales par mois

In [ ]:
orders_products = df_orders.merge(
    products_df[['product_name','sales']],
    on='product_name',
    how='left'
)
sales_per_month = orders_products.groupby('month_year')['sales'].sum().reset_index()
sales_per_month.head()

Sales par region  par période

In [ ]:
df_regions.columns

In [ ]:
sales_region_period = (
    df_orders
    .merge(products_df[['product_name','sales']], on='product_name', how='left')
    .merge(df_regions[['customer_id','region_name']], on='customer_id', how='left')
    .groupby(['month_year','region_name'])['sales']
    .sum()
    .reset_index()
)
sales_region_period.head()

Sales par category  par période

In [ ]:
orders_products = df_orders.merge(
    products_df[['product_name','sales','category']],
    on='product_name',
    how='left'
)

sales_category_period = orders_products.groupby(
    ['month_year','category']
)['sales'].sum().reset_index()

sales_category_period.head()

Top 10 Produits

In [ ]:
top_products = products_df.groupby('product_name')['sales'].sum() \
                          .sort_values(ascending=False) \
                          .head(10) \
                          .reset_index()

top_products

  Top 10 Clients

In [ ]:
orders_products = df_orders.merge(
    products_df[['product_name','sales']],
    on='product_name',
    how='left'
)

top_clients = orders_products.groupby('customer_id')['sales'].sum() \
                             .sort_values(ascending=False) \
                             .head(10) \
                             .reset_index()

top_clients

##  Création des Graphiques simples

Bar chart (Sales par catégorie)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.bar(sales_per_category['category'], sales_per_category['sales'])
plt.title("Sales par catégorie")
plt.xlabel("Category")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.show()

Line chart (Sales par mois)

In [ ]:
sales_per_month['month_year_str'] = sales_per_month['month_year'].astype(str)

import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(sales_per_month['month_year_str'], sales_per_month['sales'], marker='o')

plt.title("Sales par mois")
plt.xlabel("Month")
plt.ylabel("Sales")

# tick 
plt.xticks(sales_per_month['month_year_str'][::2], rotation=45)
plt.show()

Pie chart (Répartition des ventes par catégorie)

In [ ]:
import matplotlib.pyplot as plt

# Créer une figure de taille 7x7 pouces
plt.figure(figsize=(7,7))

# Tracer le diagramme circulaire (Pie chart)
plt.pie(
    sales_per_category['sales'],       # Valeurs des ventes par catégorie
    labels=sales_per_category['category'],  # Noms des catégories pour chaque secteur
    autopct='%1.1f%%',                # Affiche le pourcentage de chaque secteur avec 1 décimale
    startangle=90                      # Commence le premier secteur à 90° (en haut)
)

# Ajouter un titre au graphique
plt.title("Répartition des ventes par catégorie")

# Afficher le graphique
plt.show()

##  Création des Graphiques combinés

Sales vs Profit

In [ ]:
sales_profit = products_df.groupby('category')[['sales','marge']].sum().reset_index()

sales_profit.plot(
    x='category',
    y=['sales','marge'],
    kind='bar'
)

plt.title("Sales vs Profit par catégorie")
plt.show()

top produits

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.barh(top_products['product_name'], top_products['sales'])
plt.title("Top 10 produits")
plt.xlabel("Sales")
plt.show()

Heatmap avec Seaborn

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
orders_products = df_orders.merge(
    products_df[['product_name','category','sales']],
    on='product_name',
    how='left'
)
orders_with_region = orders_products.merge(
    df_regions[['customer_id','region_name']],
    on='customer_id',
    how='left'
)

# Exemple de pivot table pour heatmap
pivot_table = orders_with_region.pivot_table(
    values='sales',
    index='region_name',
    columns='category',
    aggfunc='sum'
)
#Visualisation avec Seaborn
plt.figure(figsize=(8,5))
sns.heatmap(pivot_table, annot=True, fmt=".0f", cmap="coolwarm")
plt.title("Heatmap des ventes par région et catégorie")
plt.show()
#Distribution des ventes
plt.figure(figsize=(8,5))
sns.histplot(orders_with_region['sales'], kde=True, bins=30)
plt.title("Distribution des ventes")
plt.xlabel("Sales")
plt.ylabel("Nombre de commandes")
plt.show()

Ajouter des filtres interactifs

Créer les widgets pour les filtres

In [ ]:
df_orders.columns

In [ ]:
import ipywidgets as widgets
from IPython.display import display

region_widget = widgets.Dropdown(
    options=['Tous'] + sorted(df_regions['region_name'].unique()),
    description='Région:'
)

category_widget = widgets.Dropdown(
    options=['Tous'] + sorted(products_df['category'].unique()),
    description='Catégorie:'
)
year_widget = widgets.Dropdown(
    options=['Tous'] + sorted(df_orders['month_year'].unique()),
    description='Période:'
)



Fonction pour mettre à jour les graphiques selon les filtres

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def update_graph(region, category,year):
    df_filtered = df_regions.copy()
    
    # Appliquer les filtres
    if region != 'Tous':
        df_filtered = df_filtered[df_filtered['region_name'] == region]
    if category != 'Tous':
        df_filtered = df_filtered[df_filtered['category'] == category]
    if year != 'Tous':
        df_filtered = df_filtered[df_filtered['month_year'] == year]

    
    

In [ ]:
df_orders.columns

Lier les widgets à la fonction

In [ ]:
widgets.interact(update_graph,
                region=region_widget,
                category=category_widget,
                year=year_widget,
)